# Módulo 2 — Desarrollador Full Stack de Soluciones Inteligentes

## Unidad I: Redes Neuronales y Aplicaciones Full Stack

### Clase 1: Introducción a Redes Neuronales

---



# 2. Normas y estándares aplicados para ML

---

# A. Calidad de Código en Sistemas de ML

El objetivo aquí no es “código bonito”, sino:

> Evitar modelos irreproducibles, imposibles de depurar y peligrosos en producción.

---

## 1. PEP8 aplicado a Machine Learning (no solo estilo)

En ML el incumplimiento de estilo produce bugs silenciosos.

### ❌ Incorrecto



In [ ]:
Xtrain,Xtest,Ytrain,Ytest=train_test_split(X,y,test_size=0.2)

model=Sequential([Dense(10,activation="relu"),Dense(1,activation="sigmoid")])
model.compile("adam","binary_crossentropy")


Problemas:

* variables ambiguas
* sin tipos
* difícil trazabilidad
* imposible de auditar

---


### ✔ Correcto



In [2]:
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import pandas as pd

def split_dataset(
    features: pd.DataFrame,
    target: pd.Series
) -> tuple:
    return train_test_split(
        features,
        target,
        test_size=0.2,
        random_state=42
    )


> PEP8 en ML = mantenibilidad + auditabilidad del modelo

---

## 2. Clean Code aplicado a entrenamiento

Principio:

> Un notebook no es arquitectura

---

### ❌ Antipatrón típico de IA



In [ ]:
data = pd.read_csv("data.csv")
data = clean(data)
X = data.drop("target", axis=1)
y = data["target"]

model = Sequential([...])
model.compile(...)
model.fit(X,y)
model.save("model.keras")


Esto rompe:

* SRP
* testeabilidad
* reproducibilidad

---


### ✔ Arquitectura correcta mínima

Separar responsabilidades:

```
ml_project/
│
├── data.py
├── preprocessing.py
├── model.py
├── train.py
└── evaluate.py
```

---

### Principios aplicados

| Principio | Cómo se cumple                   |
| --------- | -------------------------------- |
| SRP       | cada archivo una responsabilidad |
| DRY       | modelo reutilizable              |
| KISS      | arquitectura mínima              |
| Testeable  | se puede unit test               |

---


# B. Ciencia Reproducible (ML Engineering)

Aquí se enseñará una diferencia clave:

> Un modelo que hoy funciona, pero mañana no puede reproducirse = modelo inválido científicamente.

---


## 1. Semillas determinísticas (Fijas)

Problema: TensorFlow es estocástico.

### ❌ Sin control **Cada entrenamiento produce modelo distinto.**

### ✔ Controlado: **Esto permite:**

* debugging
* auditorías
* papers reproducibles
* producción estable

---


In [ ]:
import os
import random
import numpy as np
import tensorflow as tf

def set_global_seed(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


## 2. Versionado del dataset

Es guardar una copia exacta de los datos usados para entrenar.

Porque sin esos datos:

> Nunca podrás volver a crear ese mismo modelo.


Concepto importante:

> El modelo NO es el archivo .keras

> modelo = datos + preprocesamiento + código + parámetros + pesos


---


### Práctica recomendada

Guardar metadatos:



In [ ]:
metadata = {
    "dataset_version": "credit_v1.2",
    "features": list(X.columns),
    "train_size": len(X_train),
    "test_size": len(X_test)
}


---

## 3. Separación train/test

### El error típico

```python
scaler.fit(X)
X = scaler.transform(X)
```

Aquí estás **aprendiendo estadísticas usando TODOS los datos**, incluso los de prueba.

Entonces el modelo, indirectamente, *ya vio el examen antes de presentarlo* →
eso se llama **data leakage** (filtración de información).

El resultado: métricas artificialmente altas.

---

### Lo correcto

Primero separar:

```
datos → train + test
```

Luego:

```
fit SOLO con train
transform con train y test
```

Idea clave:

> El conjunto test debe ser completamente desconocido para el modelo.

---


In [ ]:
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)


---

## 4. ¿Qué es *Data Leakage*?

Es cuando el modelo **recibe información del futuro o del conjunto de prueba durante el entrenamiento** sin que te des cuenta.

Resultado:
el modelo parece muy bueno… pero en la realidad falla.

---

### Ejemplos típicos

**1) Escalar antes de dividir**

```python
scaler.fit(X)   # aprendió del test también
```

**2) Codificar categorías con todos los datos**

```python
encoder.fit(X_total)
```

**3) Columnas que revelan la respuesta**
Ejemplo: predecir abandono de clientes usando:

```
fecha_cancelacion
```

→ esa variable solo existe cuando el cliente YA se fue

---

### Consecuencia

Métricas infladas → modelo inútil en producción.

---

### Regla de oro

> Nada del conjunto test puede participar en nada que aprenda parámetros.


---

# C. Seguridad en Machine Learning (OWASP ML Adaptado)

> Un modelo puede ser hackeado aunque el código sea seguro.

---

## 1. Validación de datos de entrada

### ❌ Incorrecto. **Nunca confiar en input del usuario.**


```python
prediction = model.predict(user_input)
```

### ✔ Correcto

```python
def validate_features(data: list[float]) -> None:
    if len(data) != 8:
        raise ValueError("Invalid feature vector size")

    if any(x is None for x in data):
        raise ValueError("Null values detected")

    if not all(-1e6 < x < 1e6 for x in data):
        raise ValueError("Out of range values")

```


---

## 2. Dataset Poisoning (conceptual)

Es cuando alguien manipula los datos que usa el modelo para aprender, para que el modelo tome malas decisiones. No atacan el código… atacan lo que el modelo cree que es “la realidad”.


**Ataque:**
Usuarios envían datos falsos → modelo aprende comportamiento incorrecto.

**Ejemplo real:**
Sistema de crédito
usuarios simulan ingresos falsos → modelo aprueba fraude

**Regla enseñada:**

> Un modelo jamás debe reentrenarse automáticamente con datos productivos.

---



## 3. No entrenar con datos sin sanitizar

Antes de entrenar, los datos deben pasar un proceso de **validación.** *(ETL)*

Si no, el modelo aprende errores como si fueran verdades.


**Qué revisar antes de entrenar**

1) Tipos correctos: números donde deben ser números, fechas válidas, sin texto en columnas numéricas

2) Valores imposibles: edad < 0, salario negativo, porcentajes > 100%

3) Valores faltantes: null / NaN, decidir: imputar o eliminar

4) Categorías inválidas: nuevas categorías no esperadas, errores de escritura ("Colmbia", "colombia")

5) Outliers extremos: registros absurdamente grandes/pequeños, posibles fraudes o errores de captura

6) Duplicados: el mismo registro repetido muchas veces sesga el modelo

7) Etiquetas incorrectas: clases mal asignadas → el modelo aprende mal

In [ ]:
def sanitize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates()
    df = df.dropna()
    return df


---

## 4. Evitar exposición del modelo

Un modelo en producción **no debe quedar accesible libremente**, porque pueden estudiarlo hasta romperlo.

Si cualquiera puede enviar miles de consultas → puede descubrir cómo decide.

---

### Qué podría pasar

**1) Robar el modelo (Model Stealing)**
Hacen muchas preguntas y reconstruyen su comportamiento.

**2) Engañarlo (Adversarial inputs)**
Descubren qué valores cambian la predicción y lo manipulan.

**3) Forzar decisiones favorables**
Ej: ajustar datos hasta que el crédito siempre salga aprobado.

---

### Cómo se previene

* limitar número de consultas (rate limit)
* autenticación obligatoria
* no mostrar probabilidades exactas (solo aprobación/rechazo)
* registrar actividad sospechosa
* bloquear patrones automáticos

---

### Idea clave

> El modelo no solo se protege en el entrenamiento, también en cómo se consulta.


---


# Bloque 1 — Concepto computacional de neurona

---

## 1.1 ¿Qué problema intentan resolver las redes neuronales?

Antes de IA, los sistemas se programaban con reglas explícitas:

```python
if edad > 18:
    aprobar_credito()
else:
    negar_credito()
```

Eso funciona solo cuando el mundo es simple.

---

### Pero el mundo real NO funciona con reglas rígidas

En un banco real, aprobar un crédito depende de:

* ingresos
* estabilidad laboral
* historial de pagos
* edad
* número de dependientes
* tipo de contrato
* ciudad
* nivel educativo
* comportamiento financiero

Ahora intenta escribir eso con `if`…

Tendrías miles de condiciones:

```python
if ingreso > 2000 and antiguedad > 2 and deudas < 30% and ...
```

Problemas:

1. Las reglas crecen exponencialmente
2. Se contradicen entre sí
3. Se vuelven imposibles de mantener
4. No generalizan a nuevos casos

---

### Entonces aparece una necesidad computacional nueva

No queremos programar reglas.

Queremos que el sistema **aprenda patrones**.

No le decimos *qué hacer*
Le mostramos ejemplos de decisiones correctas.

Esto cambia completamente el paradigma:

| Programación clásica | Machine Learning  |
| -------------------- | ----------------- |
| reglas → respuestas  | ejemplos → patrón |
| lógica explícita     | lógica inferida   |
| programador decide   | modelo aprende    |

---

### Qué hace realmente una red neuronal

No “piensa”
No “razona”
No “entiende”

Hace algo mucho más simple:

> Aprende una función que separa casos buenos de malos.

Es decir:

Encuentra una frontera de decisión en un espacio de datos complejo.

A esto se le llama:

**aproximador universal de funciones**

(significa: puede representar comportamientos complejos sin escribir reglas)

---

## 1.2 La neurona como un tomador de decisiones muy simple

Imagina un analista de crédito extremadamente básico.

Cada característica del cliente le influye:

* ingresos → positivo
* deudas → negativo
* estabilidad → positivo

La neurona hace exactamente eso:

1. Mira todas las características
2. Les da importancia distinta
3. Produce un puntaje
4. Decide

---

### Cómo piensa una neurona (conceptualmente)

Proceso interno:

1️⃣ Observa datos
2️⃣ Calcula una puntuación
3️⃣ Aplica una regla de decisión

En código:

```python
def neuron(cliente):
    puntaje = evaluar_importancia(cliente)
    decision = aplicar_criterio(puntaje)
    return decision
```

No hay magia.
Es literalmente un sistema de puntuación aprendido.

---

### Entonces… ¿qué es realmente una red neuronal?

Una red neuronal es:

> Muchos evaluadores simples colaborando para tomar una decisión compleja.

No es una mente.

Es una cadena de filtros de información.

---

## 1.3 Componentes

---

### Peso (weight)

**Qué es:**
Cuánto importa un dato para la decisión.

Ejemplo humano:

Un banco considera más importante el historial crediticio que la edad.

Entonces:

```
historial_crediticio → peso alto
edad → peso bajo
color_favorito → peso cero
```

La red aprende estos pesos automáticamente.

---

### Bias (sesgo o ajuste)

**Qué es:**
La tendencia base del modelo.

Ejemplo:

Un banco conservador tiende a rechazar créditos.
Uno agresivo tiende a aprobarlos.

Ese “comportamiento inicial” es el bias.

Ajusta la dificultad para aprobar.

---

### Activación

**Qué es:**
La regla final de decisión.

Convierte un puntaje en una acción.

Ejemplo:

| Puntaje | Resultado |
| ------- | --------- |
| bajo    | negar     |
| medio   | evaluar   |
| alto    | aprobar   |

La activación transforma números en decisiones.

---

### Capa

**Qué es:**
Un grupo de neuronas que detectan un tipo de patrón.

Analogía visión humana:

1ª capa → detecta bordes
2ª capa → detecta formas
3ª capa → detecta objetos

Cada capa transforma información a un nivel más abstracto.

---

### Red

**Qué es:**
Varias capas trabajando juntas.

No decide de una vez.

Decide progresivamente:

datos → patrones → características → decisión



Perfecto — ahora haremos lo mismo con **activaciones y función de pérdida**:
primero la intuición (qué problema resuelven) y luego la conexión técnica.
La meta es que el estudiante entienda *por qué existen*, no solo cómo usarlas.

---

# Bloque 2 — Cómo aprende realmente una red neuronal

---

## 2.1 El problema fundamental del aprendizaje

Hasta ahora tenemos una neurona que toma decisiones.
Pero hay una pregunta crítica:

> ¿Cómo sabe la red si su decisión fue buena o mala?

Un programa normal:

```python
if respuesta == correcta:
    print("bien")
else:
    print("mal")
```

Una red neuronal no conoce la respuesta correcta internamente.
Solo recibe retroalimentación externa.

Entonces el aprendizaje ocurre así:

1. La red intenta decidir
2. Comparamos con la realidad
3. Medimos qué tan equivocada estuvo
4. Ajustamos su criterio

Ese proceso necesita **dos piezas clave**:

| Componente | Función            |
| ---------- | ------------------ |
| Activación | cómo decide        |
| Pérdida    | cuánto se equivocó |

---

# PARTE A — ACTIVACIONES

---

## ¿Por qué no basta con sumar valores?

Si la neurona solo hiciera:

```
puntaje = ingresos + estabilidad - deudas
```

Siempre produciría un número continuo infinito.

Pero las decisiones reales no son infinitas:

* aprobar / negar
* fraude / no fraude
* gato / perro

Necesitamos convertir un número en comportamiento.

Ahí aparece la activación.

---

## Qué es realmente una función de activación

> Es la personalidad de la neurona.

Transforma un puntaje interno en una reacción.

Sin activación, toda la red sería equivalente a una sola ecuación gigante.
No podría aprender comportamientos complejos.

---

## Tipos de comportamientos (intuitivo)

---

### Sigmoid — Probabilidad

La neurona responde:

> “Entre más alto el puntaje, más segura estoy”

Se usa cuando queremos:

aprobado vs rechazado

Salida tipo:

```
0.02 → casi imposible
0.50 → dudoso
0.98 → casi seguro
```

La red aprende a estimar probabilidades.

---

### Tanh — Opinión balanceada

Parecido a sigmoid pero centrado.

Representa:

* negativo
* neutral
* positivo

Útil cuando importa dirección, no solo pertenencia.

---

### ReLU — Detector de evidencia

La más importante en deep learning.

Comportamiento:

> Si no veo evidencia, no opino.
> Si veo evidencia, reacciono proporcionalmente.

Es como un sensor:

* nada relevante → silencio
* algo relevante → señal fuerte

Por eso permite aprender patrones complejos.

---

### Softmax — Elección entre opciones

No decide “sí o no”

Decide:

> cuál de todas es más probable

Ejemplo:

| Clase   | Probabilidad |
| ------- | ------------ |
| perro   | 0.70         |
| gato    | 0.20         |
| caballo | 0.10         |

Siempre suma 1 → obliga a elegir.

---

## Idea clave

La activación NO enseña
La activación define cómo puede comportarse la neurona

Es su lenguaje de respuesta.

---

# PARTE B — FUNCIÓN DE PÉRDIDA

---

## El verdadero motor del aprendizaje

La red mejora porque comete errores.

Pero necesita saber:

> qué tan grave fue su error

No basta saber si falló.
Necesitamos medir cuánto falló.

Eso es la función de pérdida.

---

## Analogía humana

Profesor corrige examen:

| Respuesta      | Evaluación           |
| -------------- | -------------------- |
| totalmente mal | gran penalización    |
| casi correcto  | pequeña penalización |
| correcto       | sin penalización     |

La pérdida es la nota negativa.

La red intenta minimizarla constantemente.

---

## Tipos de errores según el problema

---

### Problema de regresión

Predecir un número

Ejemplo:
precio de vivienda

Error:
distancia entre predicción y realidad

La red intenta acercarse lo máximo posible.

---

### Problema binario

Sí o No

Ejemplo:
fraude bancario

No importa solo equivocarse,
importa la seguridad con la que se equivocó.

Decir:
fraude 0.51 cuando era 1 → pequeño error
decir:
fraude 0.01 cuando era 1 → error grave

La pérdida penaliza la confianza incorrecta.

---

### Problema multiclase

Elegir entre muchas categorías

La red es castigada cuando asigna alta probabilidad a la clase incorrecta.

---

## Idea clave

La pérdida es el GPS del aprendizaje.

| Sin pérdida           | Con pérdida            |
| --------------------- | ---------------------- |
| no mejora             | aprende                |
| decisiones aleatorias | decisiones optimizadas |

---

# Conexión entre activación y pérdida

Este es el concepto más importante del bloque:

> La activación define el tipo de respuesta
> La pérdida evalúa si esa respuesta fue adecuada

Deben ser compatibles.

---

Ejemplo:

Si la red responde probabilidades,
la pérdida debe evaluar probabilidades.

Si responde números continuos,
la pérdida mide distancia.

---

# BLOQUE 3 — Construcción Profesional del Sistema (Arquitectura Real)

# **Mini-proyecto profesional de ML listo para producción**.
La meta: trabajar como **ML Engineer**, no como usuario de notebook.

Se usará:

> **Dataset real público: Adult Income Dataset (UCI / OpenML)**
> Predicción: si una persona gana >50K USD/año
> Tipo: clasificación binaria tabular (el caso más común en industria: scoring / riesgo / marketing)

Este dataset es estándar en cursos de ML Engineering porque:

* tiene variables categóricas y numéricas
* requiere preprocessing serio
* permite demostrar data leakage
* se comporta como problema empresarial real

---



# 1. Arquitectura del proyecto

Estructura obligatoria:

```
income-ml-system/
│
├── artifacts/              # Resultados del entrenamiento
│   ├── model.keras         # Modelo final entrenado
│   └── preprocessor.joblib # pipeline de preprocesamiento ya entrenado
│
├── data/                   # Datos fuente
│   └── raw/                # Dataset original
│
├── models/                 # Modelos experimentales
│
├── README.md               # Documentación
├── requirements.txt        # Dependencias
│
├── src/                    # Código fuente
│   ├── config.py           # Parámetros globales
│   ├── data_loader.py      # Carga datos
│   ├── evaluate.py         # Evaluación
│   ├── model.py            # Arquitectura
│   ├── preprocessing.py    # Transformaciones
│   ├── train.py            # Entrenamiento
└── └── utils.py            # Utilidades


```



---

# 2. requirements.txt (reproducible)



tensorflow==2.16.1
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.5.0
joblib==1.4.2
matplotlib==3.9.0
openml==0.14.2


Instalación:

```bash
pip install -r requirements.txt
```

---



# 3. Obtención de dataset real (NO CSV manual)

## data_loader.py

Cumple:

* reproducibilidad científica
* versionado implícito
* sin manipulación humana



In [ ]:
# Importa la librería openml, que permite descargar datasets públicos usados en Machine Learning
import openml

# Importa pandas y lo alias como pd para manipular datos tabulares (DataFrame y Series)
import pandas as pd

# Identificador del dataset dentro del repositorio OpenML
# 1590 corresponde al dataset "Adult" (predicción de ingresos >50K)
DATASET_ID = 1590  # adult dataset


# Función que carga el dataset y retorna dos estructuras:
# - X: variables predictoras (features)
# - y: variable objetivo (target)
def load_dataset() -> tuple[pd.DataFrame, pd.Series]:

    # Descarga el dataset desde OpenML usando su ID
    # Retorna un objeto Dataset con metadatos y acceso a los datos
    dataset = openml.datasets.get_dataset(DATASET_ID)

    # Extrae los datos del dataset
    # target indica cuál columna será la variable objetivo (etiqueta)
    # get_data devuelve: X, y, categorical_indicator, attribute_names
    X, y, _, _ = dataset.get_data(target=dataset.default_target_attribute)

    # Retorna las variables predictoras (DataFrame) y la variable objetivo (Series)
    return X, y


---

# 4. Configuración centralizada

## config.py

Norma: NO valores mágicos en el código



In [ ]:
SEED = 42
TEST_SIZE = 0.2

NUMERIC_FEATURES = [
    "age", "fnlwgt", "education-num", "capital-gain",
    "capital-loss", "hours-per-week"
]

CATEGORICAL_FEATURES = [
    "workclass", "education", "marital-status",
    "occupation", "relationship", "race", "sex", "native-country"
]

---

# 5. Preprocesamiento sin data leakage

## preprocessing.py

Cumple:

* OWASP ML (sanitización)
* ciencia reproducible
* separación entrenamiento / inferencia



In [ ]:
# Importa pandas (aunque aquí no se usa directamente, suele emplearse para validaciones de tipos o debugging)
import pandas as pd

# StandardScaler: estandariza variables numéricas (media=0, desviación estándar=1)
# OneHotEncoder: convierte variables categóricas en variables binarias
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ColumnTransformer permite aplicar diferentes transformaciones según el tipo de columna
from sklearn.compose import ColumnTransformer

# Pipeline permite encadenar transformaciones de manera ordenada y reproducible
from sklearn.pipeline import Pipeline

# joblib permite serializar (guardar) modelos y transformadores en disco
import joblib

# Importa la configuración del proyecto donde se definen
# las columnas numéricas y categóricas del dataset
from src.config import NUMERIC_FEATURES, CATEGORICAL_FEATURES


# Construye el objeto de preprocesamiento completo del modelo
# Retorna un ColumnTransformer listo para entrenar o inferir
def build_preprocessor() -> ColumnTransformer:

    # Pipeline para variables numéricas
    # Escala todas las columnas numéricas para mejorar el rendimiento de algoritmos ML
    numeric_pipeline = Pipeline([
        ("scaler", StandardScaler())  # Normaliza datos numéricos
    ])

    # Pipeline para variables categóricas
    # Convierte categorías en vectores binarios
    categorical_pipeline = Pipeline([
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
        # handle_unknown="ignore" evita errores si aparece una categoría nueva en producción
    ])

    # ColumnTransformer aplica cada pipeline al grupo de columnas correspondiente
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, NUMERIC_FEATURES),      # Aplica transformación numérica
        ("cat", categorical_pipeline, CATEGORICAL_FEATURES)  # Aplica transformación categórica
    ])

    # Retorna el preprocesador listo para usarse en entrenamiento o inferencia
    return preprocessor


# Guarda el preprocesador en disco para reutilizarlo en producción
def save_preprocessor(preprocessor):

    # Serializa el objeto y lo guarda en la carpeta artifacts
    # Esto permite usar EXACTAMENTE el mismo procesamiento en entrenamiento e inferencia
    joblib.dump(preprocessor, "artifacts/preprocessor.joblib")


---

# 6. Modelo (arquitectura profesional)

## model.py

Reglas aplicadas:

* entrada dinámica
* capas progresivas
* regularización básica
* reproducible



In [ ]:
# Importa TensorFlow, framework principal para construir y entrenar redes neuronales
import tensorflow as tf


# Función que construye y retorna un modelo de red neuronal compilado
# input_dim: número de variables de entrada (features) después del preprocesamiento
def build_model(input_dim: int) -> tf.keras.Model:

    # Sequential crea una red neuronal capa por capa en orden
    model = tf.keras.Sequential([

        # Define la capa de entrada
        # shape=(input_dim,) indica cuántas características recibe cada registro
        tf.keras.layers.Input(shape=(input_dim,)),

        # Primera capa densa (fully connected)
        # 64 neuronas aprenden patrones complejos no lineales
        tf.keras.layers.Dense(64, activation="relu"),

        # Normaliza las activaciones para estabilizar y acelerar el entrenamiento
        tf.keras.layers.BatchNormalization(),

        # Apaga aleatoriamente el 30% de neuronas en cada batch
        # Reduce overfitting (regularización)
        tf.keras.layers.Dropout(0.3),

        # Segunda capa densa más compacta (aprende representaciones más abstractas)
        tf.keras.layers.Dense(32, activation="relu"),

        # Nueva normalización para estabilizar gradientes
        tf.keras.layers.BatchNormalization(),

        # Dropout más suave para conservar información aprendida
        tf.keras.layers.Dropout(0.2),

        # Capa de salida
        # 1 neurona porque es clasificación binaria
        # sigmoid devuelve probabilidad entre 0 y 1
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    # Compila el modelo: define cómo aprenderá
    model.compile(

        # Adam: optimizador adaptativo basado en gradiente descendente
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),

        # Función de pérdida para clasificación binaria
        # Penaliza probabilidades incorrectas
        loss="binary_crossentropy",

        # Métricas para evaluar rendimiento del modelo
        metrics=[

            # Área bajo la curva ROC (métrica robusta para datasets desbalanceados)
            tf.keras.metrics.AUC(name="auc"),

            # Precisión: de los positivos predichos, cuántos eran correctos
            tf.keras.metrics.Precision(name="precision"),

            # Recall: de los positivos reales, cuántos detectó el modelo
            tf.keras.metrics.Recall(name="recall")
        ]
    )

    # Retorna el modelo listo para entrenamiento (.fit) o inferencia (.predict)
    return model


---

# 7. Entrenamiento correcto

## train.py

Cumple:

* determinismo
* separación de datos
* guardado de artefactos



In [ ]:
# Manejo de rutas y carpetas del sistema operativo
import os

# Librería numérica base para manejo de arrays
import numpy as np

# Framework de Deep Learning
import tensorflow as tf

# Divide el dataset en entrenamiento y prueba
from sklearn.model_selection import train_test_split

# Convierte etiquetas categóricas a numéricas
from sklearn.preprocessing import LabelEncoder

# Función personalizada que descarga/carga el dataset
from src.data_loader import load_dataset

# Construcción y guardado del pipeline de preprocesamiento
from src.preprocessing import build_preprocessor, save_preprocessor

# Construcción de la red neuronal
from src.model import build_model

# Configuraciones globales del proyecto (porcentaje test y semilla)
from src.config import TEST_SIZE, SEED


# Fija la semilla aleatoria para reproducibilidad
# Garantiza mismos resultados en cada ejecución (experimentos científicos reproducibles)
def set_seed(seed: int):
    tf.random.set_seed(seed)  # TensorFlow
    np.random.seed(seed)      # NumPy


# Función principal de entrenamiento
def train():

    # Asegura reproducibilidad
    set_seed(SEED)

    # Crea carpeta donde se guardarán modelo y preprocesador
    os.makedirs("artifacts", exist_ok=True)

    # Carga dataset (features y etiqueta)
    X, y = load_dataset()

    # TensorFlow no acepta etiquetas tipo string/category
    # Se convierten a valores numéricos (0 y 1)
    if y.dtype == "category" or y.dtype == object:
        le = LabelEncoder()                 # codificador de clases
        y = le.fit_transform(y).astype(np.float32)  # convierte a float32
    else:
        y = np.asarray(y, dtype=np.float32)  # asegura tipo numérico

    # Divide datos en entrenamiento y prueba
    # stratify=y mantiene proporción de clases (muy importante en clasificación)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )

    # Construye pipeline de transformación
    preprocessor = build_preprocessor()

    # Ajusta transformaciones SOLO con datos de entrenamiento (evita data leakage)
    X_train = preprocessor.fit_transform(X_train)

    # Aplica mismas transformaciones al conjunto de prueba
    X_test = preprocessor.transform(X_test)

    # TensorFlow requiere matrices densas y float32
    # Si OneHotEncoder produjo matriz dispersa -> convertir
    if hasattr(X_train, "toarray"):
        X_train = X_train.toarray()
    X_train = np.asarray(X_train, dtype=np.float32)

    if hasattr(X_test, "toarray"):
        X_test = X_test.toarray()
    X_test = np.asarray(X_test, dtype=np.float32)

    # Guarda el preprocesador para usar exactamente el mismo en producción
    save_preprocessor(preprocessor)

    # Construye modelo usando número final de features transformadas
    model = build_model(X_train.shape[1])

    # Entrena la red neuronal
    model.fit(
        X_train,
        y_train,
        validation_split=0.2,  # separa parte del train para validación
        epochs=30,             # número de iteraciones completas
        batch_size=64          # tamaño de lote (balance memoria/estabilidad)
    )

    # Guarda el modelo entrenado (formato moderno de TensorFlow)
    model.save("artifacts/model.keras")


# Permite ejecutar como script: python train.py
if __name__ == "__main__":
    train()


---

# 8. Evaluación profesional

## evaluate.py

No usar solo accuracy.



In [ ]:
# Permite cargar objetos serializados (preprocesador entrenado)
import joblib

# Librería numérica base
import numpy as np

# Genera métricas de clasificación (precision, recall, f1-score, soporte)
from sklearn.metrics import classification_report

# Convierte etiquetas categóricas a numéricas (igual que en entrenamiento)
from sklearn.preprocessing import LabelEncoder

# Divide el dataset en train/test (debe ser idéntico al usado en training)
from sklearn.model_selection import train_test_split

# Carga dataset original
from src.data_loader import load_dataset

# Parámetros globales del proyecto
from src.config import TEST_SIZE, SEED

# Framework de Deep Learning
import tensorflow as tf


# Función de evaluación del modelo entrenado
def evaluate():

    # Carga datos crudos nuevamente
    X, y = load_dataset()

    # Debe repetirse EXACTAMENTE la misma codificación de etiquetas usada en train
    # Si no, las métricas serían incorrectas
    if y.dtype == "category" or y.dtype == object:
        le = LabelEncoder()
        y = le.fit_transform(y)
    else:
        y = np.asarray(y)

    # Se reconstruye el mismo split que en entrenamiento
    # Gracias al mismo random_state y stratify
    _, X_test, _, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )

    # Carga el preprocesador entrenado (NO se vuelve a entrenar)
    preprocessor = joblib.load("artifacts/preprocessor.joblib")

    # Carga el modelo entrenado
    model = tf.keras.models.load_model("artifacts/model.keras")

    # Aplica transformaciones al test igual que en training
    X_test = preprocessor.transform(X_test)

    # Convierte matriz dispersa a densa si OneHotEncoder la generó
    if hasattr(X_test, "toarray"):
        X_test = X_test.toarray()

    # TensorFlow requiere float32
    X_test = np.asarray(X_test, dtype=np.float32)

    # Predice probabilidades -> convierte a clases binarias usando umbral 0.5
    preds = (model.predict(X_test) > 0.5).astype(int).ravel()

    # Muestra reporte completo de métricas
    print(classification_report(y_test, preds))


# Permite ejecutar como script independiente: python evaluate.py
if __name__ == "__main__":
    evaluate()


---

# Flujo correcto del sistema (lo que deben entender)

Orden obligatorio:

1. Obtener datos
2. Separar train/test
3. Ajustar preprocessing SOLO en train
4. Transformar test
5. Entrenar modelo
6. Guardar artefactos
7. Evaluar modelo congelado

Nunca:

```
fit → split → evaluar
```

Siempre:

```
split → fit(train) → transform(test) → entrenar
```



---

## Cómo ejecutarlo

### 1. Instalar dependencias

`pip install -r requirements.txt`

### 2. Entrenar el modelo

Desde la **raíz del proyecto** (donde está `src/` y `requirements.txt`):

`python -m src.train`

Esto descarga el dataset, entrena el modelo y guarda en `artifacts/`:

- `artifacts/model.keras`
- `artifacts/preprocessor.joblib`

La primera vez puede tardar un poco por la descarga del dataset y el entrenamiento.

### 3. Evaluar el modelo

Después de entrenar:

`python -m src.evaluate`

Muestra el **classification report** (precision, recall, F1, etc.) en el conjunto de test.

---

## Orden de ejecución

1. `python -m src.train` → primero.
2. `python -m src.evaluate` → después (necesita los archivos en `artifacts/`).

---